[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Advanced-Time-Series/blob/main/Quantlets/Ch_02/ATS_ch2_nonlinearity_tests/ATS_ch2_nonlinearity_tests.ipynb)

# ATS_ch2_nonlinearity_tests

A battery of nonlinearity tests (BDS, Keenan, Tsay, Terasvirta LM3, Hansen sup-Wald) on five series: lynx, US unemployment changes, Romanian GDP growth, the real dollar-sterling rate and daily EUR/RON returns.

*Advanced Time Series Analysis and Forecasting (ATS), Chapter 2: Structural breaks and nonlinear models. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/ATS/Chapter_2'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import optimize
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)
LightBlue = '#7EA6E0'         # thin background series (e.g. the other countries of a panel): a blue tint, never grey

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import ats_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, MainBlue=MainBlue, IDAred=IDAred, Forest=Forest, Amber=Amber,
                           Orange=Orange, Purple=Purple, Teal=Teal, Crimson=Crimson, DarkText=DarkText, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/Advanced-Time-Series/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
# local copy of the Oxford-Man realized library, if present (not distributed; read_omi downloads it)
REALIZED_DIR = next((os.path.join(d, 'data', 'realized') for d in ('.', '..', '../..', '../../..')
                     if os.path.isdir(os.path.join(d, 'data', 'realized'))), '')
END = '2026-09-18'          # last day of the saved data
# Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and Sheppard 2009): the last public
# file (28 February 2022), preserved by the Internet Archive; six indices and the main measures
OMI_URL = ('https://web.archive.org/web/20220301022212id_/'
           'https://realized.oxford-man.ox.ac.uk/images/oxfordmanrealizedvolatilityindices.zip')
OMI_FILE = 'omi_realized_library_v03.csv'
OMI_SYMBOLS = ['.SPX', '.GDAXI', '.FTSE', '.N225', '.STOXX50E', '.FCHI']
OMI_COLS = ['rv5', 'rv5_ss', 'rv10', 'bv', 'bv_ss', 'medrv', 'rk_parzen', 'rk_twoscale', 'rsv', 'open_to_close',
            'close_price', 'open_price', 'nobs']

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the ATS repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_omi(symbols=None, local=True):
    """Oxford-Man Institute's realized library, version 0.3: daily realised measures (OMI_COLS) of six indices
    (OMI_SYMBOLS), columns date, symbol and the measures, sorted by symbol and date. The library is not
    redistributed with the course: a local copy data/realized/omi_realized_library_v03.csv is used if present,
    otherwise the last public file (Internet Archive copy of 28 February 2022) is downloaded once, the six indices
    are extracted and cached in the temporary folder. Cite: Heber, G., Lunde, A., Shephard, N. and Sheppard, K.
    (2009), Oxford-Man Institute's realized library, version 0.3, Oxford-Man Institute, University of Oxford."""
    import tempfile
    import zipfile
    if 'omi' not in _CACHE:
        cache = os.path.join(tempfile.gettempdir(), 'ats_omi')
        cands = ([os.path.join(REALIZED_DIR, OMI_FILE)] if (local and REALIZED_DIR) else []) + [os.path.join(cache, OMI_FILE)]
        path = next((c for c in cands if os.path.exists(c)), '')
        if not path:
            print("Downloading the Oxford-Man Institute's realized library, version 0.3 (Heber, Lunde, Shephard and "
                  "Sheppard 2009), last public file of 28 February 2022, from the Internet Archive")
            os.makedirs(cache, exist_ok=True)
            zpath = os.path.join(cache, 'oxfordmanrealizedvolatilityindices.zip')
            if not os.path.exists(zpath):
                req = urllib.request.Request(OMI_URL, headers={'User-Agent': 'Mozilla/5.0'})
                with urllib.request.urlopen(req, timeout=300) as r:
                    raw = r.read()
                with open(zpath, 'wb') as f:
                    f.write(raw)
            with zipfile.ZipFile(zpath) as z:
                d = pd.read_csv(z.open(z.namelist()[0]))
            d = d.rename(columns={d.columns[0]: 'date'})
            d['date'] = pd.to_datetime(d['date'].str[:10])
            d = d[d['Symbol'].isin(OMI_SYMBOLS)][['date', 'Symbol'] + OMI_COLS].rename(columns={'Symbol': 'symbol'})
            path = os.path.join(cache, OMI_FILE)
            d.sort_values(['symbol', 'date']).to_csv(path, index=False, float_format='%.6g')
        _CACHE['omi'] = pd.read_csv(path, parse_dates=['date'])
    d = _CACHE['omi']
    if symbols is not None:
        d = d[d['symbol'].isin([symbols] if isinstance(symbols, str) else list(symbols))]
    return d.copy()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years


def read_fred(ids, start=None, end=None):
    """FRED series (St. Louis Fed) from the public fredgraph CSV, e.g. read_fred('UNRATE') or
    read_fred(['GDPC1', 'CPIAUCSL']). Returns a Series (one id) or a DataFrame (several ids)."""
    one = isinstance(ids, str)
    ids = [ids] if one else list(ids)
    out = []
    for sid in ids:
        key = ('fred', sid)
        if key not in _CACHE:
            url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}'
            t = pd.read_csv(url, index_col=0, parse_dates=True, na_values='.')
            _CACHE[key] = pd.to_numeric(t.iloc[:, 0], errors='coerce').rename(sid)
        out.append(_CACHE[key])
    df = pd.concat(out, axis=1).loc[start:end]
    df.index.name = 'date'
    return df.iloc[:, 0].dropna() if one else df


def _eurostat_period(p):
    p = str(p)
    if '-Q' in p:
        y, q = p.split('-Q')
        return pd.Timestamp(int(y), 3 * int(q) - 2, 1)
    if '-' in p and len(p) == 7:
        return pd.Timestamp(p + '-01')
    if len(p) == 4:
        return pd.Timestamp(int(p), 1, 1)
    return pd.Timestamp(p)


def read_eurostat(dataset, key, start=None):
    """One Eurostat series from the public SDMX 2.1 API (SDMX-CSV), e.g. Romanian quarterly real GDP,
    seasonally and calendar adjusted, chain-linked volumes (2010) in million EUR:
        read_eurostat('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')
    `key` is the dot-separated series key of the dataset (dimensions in the order of the dataset)."""
    url = (f'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/{dataset}/{key}?format=SDMX-CSV'
           + (f'&startPeriod={start}' if start else ''))
    ck = ('eurostat', url)
    if ck not in _CACHE:
        t = pd.read_csv(url)
        s = pd.Series(pd.to_numeric(t['OBS_VALUE'], errors='coerce').values,
                      index=[_eurostat_period(p) for p in t['TIME_PERIOD']], name=f'{dataset}:{key}')
        s.index.name = 'date'
        _CACHE[ck] = s.sort_index().dropna()
    return _CACHE[ck]


def load_statsmodels(name):
    """Classic data sets shipped with statsmodels (offline): 'sunspots' (yearly, 1700-2008), 'co2' (weekly,
    Mauna Loa, 1958-2001), 'macrodata' (US quarterly macro, 1959-2009), 'nile' (yearly Nile flow, 1871-1970)."""
    import statsmodels.api as sm
    if name == 'sunspots':
        d = sm.datasets.sunspots.load_pandas().data
        s = d.set_index(pd.to_datetime(d['YEAR'].astype(int).astype(str)))['SUNACTIVITY']
        s.index.name = 'date'
        return s.rename('sunspots')
    if name == 'co2':
        return sm.datasets.co2.load_pandas().data['co2'].rename('co2')
    if name == 'macrodata':
        d = sm.datasets.macrodata.load_pandas().data
        d.index = pd.period_range('1959Q1', periods=len(d), freq='Q').to_timestamp()
        d.index.name = 'date'
        return d
    if name == 'nile':
        d = sm.datasets.nile.load_pandas().data
        s = d.set_index(pd.to_datetime(d['year'].astype(int).astype(str)))['volume']
        s.index.name = 'date'
        return s.rename('nile')
    raise KeyError(f'unknown statsmodels data set: {name}')

## Analysis

In [ ]:
import io

import zipfile

import urllib.error

import statsmodels.api as sm

HERE = "."

SEED = 2026

TRIM = 0.15

HICP_RO = ('prc_hicp_minr', 'M.RCH_A.TOTAL.RO')

GDP_RO = ('namq_10_gdp', 'Q.CLV10_MEUR.SCA.B1GQ.RO')

BP_URL = 'http://qed.econ.queensu.ca/jae/2003-v18.1/bai-perron/bp-data.zip'

MPQ_SAMPLE = ('1953-04-01', '1999-04-01')

HANSEN_SAMPLE = ('1959-01-01', '1996-07-01')

VDTF_SAMPLE = ('1968-06-01', '1999-12-01', '1989-12-01')

TPS_SAMPLE = ('1973-01-01', '1996-12-01')

UK_CPI_ONS = 'https://www.ons.gov.uk/generator?format=csv&uri=/economy/inflationandpriceindices/timeseries/d7bt/mm23'

MON_HIST = ('2015-01-01', '2019-12-01')

LYNX_SPEC = (7, 2, 2)

LWZ_C0, LWZ_D0 = 0.299, 0.1

A2_5 = 7.78

_FILES = {}

_NULL = {}

# Bai-Perron critical values (simulated once with bp_critical(T=1000, reps=1000); see generate_all_charts.py)

_BPCV = {'cv': {"T": 1000, "reps": 1000, "h": 150, "0.10": {"supF": [7.322371381025062, 6.344773604508686, 5.254470804906281, 4.405483410260758, 3.323999049482492], "UDmax": 7.688299359394575, "WDmax": 8.654388747387188, "seq": [8.727514878572347, 9.548329026081907, 10.527401722365736, 11.372495761411466]}, "0.05": {"supF": [8.73565483415872, 7.269454777472157, 6.052808536671888, 4.905247793840771, 3.8113917106752906], "UDmax": 8.865985232666754, "WDmax": 10.135588137265033, "seq": [10.537088133015938, 11.559607468696287, 11.956268366315074, 13.285584261596563]}, "0.01": {"supF": [13.302833718226006, 8.59275828135307, 7.487615986807845, 6.114998121437082, 4.641838995948307], "UDmax": 13.303865602045159, "WDmax": 15.085966620369474, "seq": [14.466784827343213, 15.044271571989226, 15.151445171680374, 15.21933654088081]}}}

def get_bytes(url):
    """Download a public file once per session (no key); optional local cache folder ATS_CACHE."""
    import hashlib
    import time
    if url in _FILES:
        return _FILES[url]
    cache = os.environ.get('ATS_CACHE')
    path = os.path.join(cache, hashlib.md5(url.encode()).hexdigest()) if cache else None
    if path and os.path.exists(path):
        _FILES[url] = open(path, 'rb').read()
        return _FILES[url]
    for attempt in range(5):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (ATS course)'})
            _FILES[url] = urllib.request.urlopen(req, timeout=180).read()
            break
        except urllib.error.URLError:
            if attempt == 4:
                raise
            time.sleep(5 * (attempt + 1))
    if path:
        os.makedirs(cache, exist_ok=True)
        open(path, 'wb').write(_FILES[url])
    return _FILES[url]


def save(name, save_it=True):
    if save_it:
        st.check_no_grey(plt.gcf())
        st.save_fig(name)
    else:
        plt.show()


def bp_real_rate():
    """US ex-post real interest rate, quarterly 1961Q1-1986Q3 (Garcia and Perron 1996), as used by Bai and Perron (2003)."""
    z = zipfile.ZipFile(io.BytesIO(get_bytes(BP_URL)))
    v = np.array([float(x) for x in z.read('real.dat').decode().split()])
    return pd.Series(v, index=pd.period_range('1961Q1', periods=len(v), freq='Q').to_timestamp(), name='real rate')


def us_real_rate():
    """Ex-post real rate rebuilt from FRED, 1961Q1 to the last quarter: the 3-month T-bill rate in the last month of quarter
    t (TB3MS) minus the annualised CPI inflation of quarter t + 1 (CPIAUCNS, last month of each quarter), in %. Over
    1961Q1-1986Q3 this construction has a correlation of 0.99 with the Bai-Perron series."""
    tb = read_fred('TB3MS').resample('QS').last()
    c = read_fred('CPIAUCNS')
    n = c.resample('QS').count()
    p = c.resample('QS').last()[n == 3]
    infl = 400 * np.log(p).diff()
    r = (tb - infl.shift(-1)).dropna().loc['1961-01-01':]
    return r.rename('real rate')


def ro_hicp(start='2000-01-01'):
    """Romanian HICP inflation, annual rate of change, monthly, % (Eurostat)."""
    return read_eurostat(*HICP_RO).loc[start:].rename('HICP RO')


def us_gdp_growth():
    """US real GDP growth, quarterly, 100 x log difference (FRED GDPC1)."""
    return (100 * np.log(read_fred('GDPC1')).diff()).dropna().rename('GDP growth')


def unemp_men(sa=True):
    """US unemployment rate of men aged 20 and over, monthly, %, computed as the ratio of the unemployment level to the
    civilian labour force of this group (BLS series via FRED: LNS13000025 / LNS11000025 seasonally adjusted,
    LNU03000025 / LNU01000025 not adjusted), the construction of Hansen (1997) and van Dijk et al. (2002)."""
    u, lf = ('LNS13000025', 'LNS11000025') if sa else ('LNU03000025', 'LNU01000025')
    return (100 * read_fred(u) / read_fred(lf)).dropna().rename('unemployment men 20+')


def uk_cpi():
    """UK CPI, all items, 2015 = 100: OECD Main Economic Indicators (FRED GBRCPIALLMINMEI) to 1987, then ONS series D7BT
    (CPI index 00, monthly from 1988), spliced at January 1988."""
    mei = read_fred('GBRCPIALLMINMEI')
    t = pd.read_csv(io.BytesIO(get_bytes(UK_CPI_ONS)), header=None, names=['p', 'v'])
    t = t[t['p'].astype(str).str.match(r'^\d{4} [A-Z]{3}$')]
    ons = pd.Series(pd.to_numeric(t['v']).values, index=pd.to_datetime(t['p'], format='%Y %b')).sort_index()
    k = ons.loc['1988-01-01'] / mei.loc['1988-01-01']
    return pd.concat([k * mei.loc[:'1987-12-01'], ons.loc['1988-01-01':]]).rename('UK CPI')


def real_usd_gbp(start=TPS_SAMPLE[0], end=None):
    """Log real dollar-sterling rate q = s + p* - p (s: dollars per pound at the end of the month, FRED DEXUSUK; p: US CPI
    not seasonally adjusted, CPIAUCNS; p*: UK CPI), normalised so that q(1973M01) = 0, as in Taylor, Peel and Sarno (2001)."""
    s = read_fred('DEXUSUK').dropna().resample('MS').last()
    q = (np.log(s) + np.log(uk_cpi()) - np.log(read_fred('CPIAUCNS'))).dropna().loc[start:end]
    return (q - q.iloc[0]).rename('q')


def ro_reer():
    """Romania, real effective exchange rate (broad, CPI-based), monthly, BIS via FRED RBROBIS; log, 2020 = 0."""
    r = np.log(read_fred('RBROBIS').dropna())
    return (r - r.loc['2020'].mean()).rename('REER RO')


def lynx():
    """Canadian lynx trappings, 1821-1934 (Elton and Nicholson 1942), the R data set read by statsmodels; log10."""
    import statsmodels.api as sm
    d = sm.datasets.get_rdataset('lynx').data
    return pd.Series(np.log10(d['value'].values), index=d['time'].astype(int).values, name='log10 lynx')


def qlabel(ts):
    p = pd.Timestamp(ts).to_period('Q')
    return f'{p.year}Q{p.quarter}'


def mlabel(ts):
    return str(pd.Timestamp(ts))[:7]


def ols(y, X):
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    return b, e, float(e @ e)


def lags(x, p, const=True):
    """Matrix [1, x_{t-1}, ..., x_{t-p}] and the target x_t (numpy arrays, first p rows dropped)."""
    x = np.asarray(x, float)
    T = len(x)
    X = np.column_stack([x[p - j:T - j] for j in range(1, p + 1)]) if p else np.empty((T, 0))
    if const:
        X = np.column_stack([np.ones(T - p), X])
    return x[p:], X


def lrv_qs(u, prewhite=True):
    """Long-run variance of a scalar series: Quadratic Spectral kernel, Andrews (1991) AR(1) plug-in bandwidth, with AR(1)
    prewhitening (Andrews and Monahan 1992), the default of Bai and Perron (2003)."""
    u = np.asarray(u, float) - np.mean(u)
    T = len(u)
    rho = 0.0
    if prewhite and T > 3:
        rho = float(u[1:] @ u[:-1] / (u[:-1] @ u[:-1]))
        rho = max(min(rho, 0.97), -0.97)
        u = u[1:] - rho * u[:-1]
        T = len(u)
    a = float(u[1:] @ u[:-1] / (u[:-1] @ u[:-1])) if T > 3 else 0.0
    a = max(min(a, 0.97), -0.97)
    alpha2 = 4 * a ** 2 / (1 - a) ** 4
    bw = 1.3221 * (alpha2 * T) ** 0.2 if alpha2 > 0 else 0.0
    v = u @ u / T
    if bw > 0:
        for j in range(1, T):
            x = 6 * np.pi * (j / bw) / 5
            w = 25 / (12 * np.pi ** 2 * (j / bw) ** 2) * (np.sin(x) / x - np.cos(x))
            g = u[j:] @ u[:-j] / T
            v += 2 * w * g
            if j > 20 * bw + 50:
                break
    return float(v / (1 - rho) ** 2)


def nw_lrv(u, L=None):
    """Bartlett (Newey-West) long-run variance with the lag 4 (T/100)^(2/9) if L is None."""
    u = np.asarray(u, float) - np.mean(u)
    T = len(u)
    L = int(np.floor(4 * (T / 100) ** (2 / 9))) if L is None else L
    v = u @ u / T
    for k in range(1, L + 1):
        v += 2 * (1 - k / (L + 1)) * (u[k:] @ u[:-k]) / T
    return float(v)


def recursive_residuals(y, X):
    """Standardised recursive residuals w_t = (y_t - x_t' b_{t-1}) / sqrt(1 + x_t'(X_{t-1}'X_{t-1})^{-1} x_t), t = k+1..T
    (Brown, Durbin and Evans 1975)."""
    T, k = X.shape
    w = np.full(T, np.nan)
    for t in range(k, T):
        Xt, yt = X[:t], y[:t]
        A = np.linalg.pinv(Xt.T @ Xt)
        b = A @ Xt.T @ yt
        w[t] = (y[t] - X[t] @ b) / np.sqrt(1 + X[t] @ A @ X[t])
    return w


def dm_hln(d, h=1):
    """Diebold-Mariano with the Harvey-Leybourne-Newbold correction for a loss differential d (h - 1 rectangular lags),
    Student t(T - 1) p-value (two-sided)."""
    d = np.asarray(d, float)
    d = d[~np.isnan(d)]
    T = len(d)
    v = np.var(d)
    x = d - d.mean()
    for k in range(1, h):
        v += 2 * (x[k:] @ x[:-k]) / T
    dm = d.mean() / np.sqrt(v / T)
    hln = dm * np.sqrt((T + 1 - 2 * h + h * (h - 1) / T) / T)
    return {'T': T, 'dbar': float(d.mean()), 'hln': float(hln), 'p': float(2 * stats.t.sf(abs(hln), T - 1))}


def break_wald_seq(y, X, trim=TRIM, robust='hc', idx=None):
    """Wald statistics for a break in the coefficients idx (default: all) at every date in the trimmed range:
    y = X b + (X_idx 1{t > k}) d + e, robust variance (hc: White; hac: QS prewhitened, scalar case only; none)."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    T, p = X.shape
    idx = list(range(p)) if idx is None else idx
    ks = np.arange(int(np.floor(trim * T)), int(np.ceil((1 - trim) * T)) + 1)
    W = np.full(len(ks), np.nan)
    for i, k in enumerate(ks):
        D = X[:, idx] * (np.arange(T) >= k)[:, None]
        Z = np.column_stack([X, D])
        b, e, ssr = ols(y, Z)
        A = np.linalg.pinv(Z.T @ Z)
        if robust == 'hc':
            V = A @ (Z.T * e ** 2) @ Z @ A
        elif robust == 'hac':
            S = np.zeros((Z.shape[1], Z.shape[1]))
            g = Z * e[:, None]
            L = int(np.floor(4 * (T / 100) ** (2 / 9)))
            S = g.T @ g
            for l in range(1, L + 1):
                G = g[l:].T @ g[:-l]
                S += (1 - l / (L + 1)) * (G + G.T)
            V = A @ S @ A
        else:
            V = A * ssr / (T - Z.shape[1])
        d = b[p:]
        Vd = V[p:, p:]
        W[i] = float(d @ np.linalg.solve(Vd, d))
    return ks, W


def bb_null(p=1, trim=TRIM, reps=20000, grid=2000, seed=SEED):
    """Simulated asymptotic null distribution of the sup, exp and ave Wald statistics (p restrictions): the Wald process
    converges to |B(r) - r B(1)|^2 / (r(1 - r)), B a p-dimensional Brownian motion (Andrews 1993)."""
    rng = np.random.default_rng(seed + p)
    r = np.arange(1, grid + 1) / grid
    keep = (r >= trim) & (r <= 1 - trim)
    sup, exp_, ave = [], [], []
    for _ in range(reps // 1000):
        B = np.cumsum(rng.standard_normal((1000, grid, p)), axis=1) / np.sqrt(grid)
        BB = B - r[None, :, None] * B[:, -1:, :]
        Q = (BB ** 2).sum(axis=2)[:, keep] / (r[keep] * (1 - r[keep]))[None, :]
        sup.append(Q.max(axis=1))
        exp_.append(np.log(np.mean(np.exp(Q / 2), axis=1)))
        ave.append(Q.mean(axis=1))
    return {k: np.concatenate(v) for k, v in (('sup', sup), ('exp', exp_), ('ave', ave))}


def null_dist(p=1, trim=TRIM):
    key = (p, trim)
    if key not in _NULL:
        _NULL[key] = bb_null(p, trim)
    return _NULL[key]


def andrews_tests(y, X, trim=TRIM, robust='hc', idx=None):
    """sup-Wald (Andrews 1993), exp-Wald and ave-Wald (Andrews and Ploberger 1994) with simulated asymptotic p-values."""
    ks, W = break_wald_seq(y, X, trim, robust, idx)
    p = X.shape[1] if idx is None else len(idx)
    nd = null_dist(p, trim)
    out = {'sup': float(W.max()), 'exp': float(np.log(np.mean(np.exp(W / 2)))), 'ave': float(W.mean()),
           'k_hat': int(ks[np.argmax(W)])}
    for s in ('sup', 'exp', 'ave'):
        out[f'p_{s}'] = float(np.mean(nd[s] >= out[s]))
    out['cv_sup'] = float(np.quantile(nd['sup'], 0.95))
    return out, ks, W


def segment_ssr(y, X, h):
    """SSR of the regression of y on X over every segment [i, j] (0-based, inclusive) with at least h observations;
    np.inf otherwise. Uses cumulative sums of X'X and X'y (pure structural change)."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    T, q = X.shape
    Cxx = np.concatenate([np.zeros((1, q, q)), np.cumsum(X[:, :, None] * X[:, None, :], axis=0)])
    Cxy = np.concatenate([np.zeros((1, q)), np.cumsum(X * y[:, None], axis=0)])
    Cyy = np.concatenate([[0.0], np.cumsum(y ** 2)])
    S = np.full((T, T), np.inf)
    if q == 1 and np.allclose(X, 1.0):                  # mean-shift model: closed form
        for i in range(T - h + 1):
            j = np.arange(i + h - 1, T)
            sy = Cxy[j + 1, 0] - Cxy[i, 0]
            S[i, j] = Cyy[j + 1] - Cyy[i] - sy ** 2 / (j - i + 1)
        return S
    for i in range(T - h + 1):
        j = np.arange(i + h - 1, T)
        A = Cxx[j + 1] - Cxx[i]
        c = Cxy[j + 1] - Cxy[i]
        b = np.linalg.solve(A + 1e-12 * np.eye(q)[None], c[:, :, None])[:, :, 0]
        S[i, j] = Cyy[j + 1] - Cyy[i] - np.einsum('ij,ij->i', b, c)
    return S


def dp_breaks(S, M, h):
    """Global minimisers of the SSR for m = 0..M breaks by dynamic programming (Bai and Perron 2003, Section 3).
    Returns {m: (SSR, [last observation of each regime except the last])}."""
    T = S.shape[0]
    out = {0: (float(S[0, T - 1]), [])}
    # V[m][j]: minimal SSR of the first j+1 observations with m breaks; P[m][j]: position of the last break
    V = {0: S[0].copy()}
    P = {}
    for m in range(1, M + 1):
        V[m] = np.full(T, np.inf)
        P[m] = np.full(T, -1)
        for j in range((m + 1) * h - 1, T):
            cand = np.arange(m * h - 1, j - h + 1)
            if len(cand) == 0:
                continue
            tot = V[m - 1][cand] + S[cand + 1, j]
            a = int(np.argmin(tot))
            V[m][j], P[m][j] = tot[a], cand[a]
        if not np.isfinite(V[m][T - 1]):
            break
        br, j = [], T - 1
        for mm in range(m, 0, -1):
            j = int(P[mm][j])
            br.append(j)
        out[m] = (float(V[m][T - 1]), sorted(br))
    return out


def segments(T, br):
    edges = [-1] + list(br) + [T - 1]
    return [(edges[i] + 1, edges[i + 1]) for i in range(len(edges) - 1)]


def bp_fstat(y, X, br, robust=True):
    """sup F statistic for given break dates (pure change in all coefficients): Wald / (k q), with per-segment
    long-run variances (heterogeneous variances and serial correlation allowed, QS prewhitened) if robust."""
    y, X = np.asarray(y, float), np.asarray(X, float)
    T, q = X.shape
    segs = segments(T, br)
    k = len(br)
    bs, Vs = [], []
    for a, b in segs:
        Xs, ys = X[a:b + 1], y[a:b + 1]
        bb, e, ssr = ols(ys, Xs)
        A = np.linalg.inv(Xs.T @ Xs)
        if robust and q == 1:
            Vs.append(A * lrv_qs(e) * len(e) * A)
        elif robust:
            Vs.append(A @ (Xs.T * e ** 2) @ Xs @ A)
        else:
            Vs.append(None)
        bs.append(bb)
    if not robust:
        e_all = np.concatenate([y[a:b + 1] - X[a:b + 1] @ bs[i] for i, (a, b) in enumerate(segs)])
        s2 = e_all @ e_all / (T - (k + 1) * q)
        Vs = [np.linalg.inv(X[a:b + 1].T @ X[a:b + 1]) * s2 for a, b in segs]
    d = np.concatenate([bs[i + 1] - bs[i] for i in range(k)])
    R = np.zeros((k * q, (k + 1) * q))
    for i in range(k):
        R[i * q:(i + 1) * q, i * q:(i + 1) * q] = -np.eye(q)
        R[i * q:(i + 1) * q, (i + 1) * q:(i + 2) * q] = np.eye(q)
    Vfull = np.zeros(((k + 1) * q, (k + 1) * q))
    for i, V in enumerate(Vs):
        Vfull[i * q:(i + 1) * q, i * q:(i + 1) * q] = V
    W = float(d @ np.linalg.solve(R @ Vfull @ R.T, d))
    return W / (k * q)


def bp_seq_stat(y, X, br, h, robust=True):
    """sup F(l+1 | l): the largest single-break statistic within the l + 1 segments of the l-break model."""
    T = len(y)
    best, where = 0.0, None
    for a, b in segments(T, br):
        n = b - a + 1
        if n < 2 * h:
            continue
        for k in range(a + h - 1, b - h + 1):
            f = bp_fstat(y[a:b + 1], X[a:b + 1], [k - a], robust)
            if f > best:
                best, where = f, k
    return best, where


def bp_critical(q=1, trim=TRIM, M=5, T=200, reps=1500, seed=SEED):
    """Critical values of sup F(k), UDmax, WDmax and sup F(l+1|l) for a pure mean-shift (q = 1) by simulation of the
    statistics on white noise (homoskedastic version, T = 200); BP (1998, 2003) tabulate the asymptotic values."""
    rng = np.random.default_rng(seed)
    h = int(np.floor(trim * T))
    X = np.ones((T, q))
    F = np.zeros((reps, M))
    for r in range(reps):
        y = rng.standard_normal(T)
        S = segment_ssr(y, X, h)
        res = dp_breaks(S, M, h)
        for m in range(1, M + 1):
            if m in res:
                s2 = res[m][0] / (T - (m + 1) * q)
                F[r, m - 1] = (res[0][0] - res[m][0]) / s2 / (m * q)
    out = {'T': T, 'reps': reps, 'h': h}
    for a in (0.10, 0.05, 0.01):
        cv = np.quantile(F, 1 - a, axis=0)
        ud = np.quantile(F.max(axis=1), 1 - a)
        wd = np.quantile((F * (cv[0] / cv)[None, :]).max(axis=1), 1 - a)
        sup1 = F[:, 0]
        seq = [float(np.quantile(sup1, (1 - a) ** (1 / (l + 1)))) for l in range(1, M)]
        out[f'{a:.2f}'] = {'supF': cv.tolist(), 'UDmax': float(ud), 'WDmax': float(wd), 'seq': seq}
    return out


def bai_ci_quantiles(phi=1.0, xi=1.0, reps=4000, S=200.0, step=0.05, seed=SEED):
    """Quantiles of argmax_s Z(s), Z(s) = W1(-s) - |s|/2 for s <= 0 and sqrt(phi) W2(s) - xi s/2 for s > 0 (Bai 1997),
    by simulation on a grid; returns the 2.5% and 97.5% quantiles (and 5%, 95%)."""
    rng = np.random.default_rng(seed)
    n = int(S / step)
    s = np.arange(1, n + 1) * step
    out = []
    for _ in range(reps // 500):
        W1 = np.cumsum(rng.standard_normal((500, n)) * np.sqrt(step), axis=1) - s / 2
        W2 = np.sqrt(phi) * np.cumsum(rng.standard_normal((500, n)) * np.sqrt(step), axis=1) - xi * s / 2
        left = W1.max(axis=1)
        right = W2.max(axis=1)
        am = np.where(left > np.maximum(right, 0), -s[W1.argmax(axis=1)], np.where(right > 0, s[W2.argmax(axis=1)], 0.0))
        out.append(am)
    a = np.concatenate(out)
    return {k: float(np.quantile(a, v)) for k, v in (('q025', 0.025), ('q05', 0.05), ('q95', 0.95), ('q975', 0.975))}


def bp_analysis(y, M=5, trim=TRIM, cv=None, robust=True, ci=True):
    """Full Bai-Perron analysis of a mean-shift model: breaks for m = 1..M, sup F(k), UDmax, sequential selection at 5%,
    BIC and LWZ, segment means and 95% confidence intervals for the break dates (Bai 1997, heterogeneous long-run
    variances across segments)."""
    y = np.asarray(y, float)
    T = len(y)
    X = np.ones((T, 1))
    h = int(np.floor(trim * T))
    S = segment_ssr(y, X, h)
    res = dp_breaks(S, M, h)
    M = max(res)
    out = {'T': T, 'h': h, 'ssr': {m: res[m][0] for m in res}, 'breaks': {m: res[m][1] for m in res}}
    out['supF'] = {m: bp_fstat(y, X, res[m][1], robust) for m in range(1, M + 1)}
    out['UDmax'] = max(out['supF'].values())
    out['BIC'] = {m: float(np.log(res[m][0] / T) + (m + 1 + m) * np.log(T) / T) for m in res}
    out['LWZ'] = {m: float(np.log(res[m][0] / (T - (m + 1 + m))) + (m + 1 + m) * LWZ_C0 * np.log(T) ** (2 + LWZ_D0) / T)
                  for m in res}
    out['m_bic'] = int(min(out['BIC'], key=out['BIC'].get))
    out['m_lwz'] = int(min(out['LWZ'], key=out['LWZ'].get))
    seq = {}
    m_seq = 0
    if cv is not None and out['supF'][1] > cv['0.05']['supF'][0]:
        m_seq = 1
        for l in range(1, M):
            f, _ = bp_seq_stat(y, X, res[l][1], h, robust)
            seq[l] = f
            if f > cv['0.05']['seq'][l - 1]:
                m_seq = l + 1
            else:
                break
    out['seq'] = seq
    out['m_seq'] = m_seq
    return out


def bp_fit(y, br, ci=True):
    """Segment means, long-run variances and 95% Bai (1997) confidence intervals for given mean-shift break dates."""
    y = np.asarray(y, float)
    T = len(y)
    segs = segments(T, br)
    mu = [float(y[a:b + 1].mean()) for a, b in segs]
    lrv = [lrv_qs(y[a:b + 1] - y[a:b + 1].mean()) for a, b in segs]
    cis = []
    if ci:
        for i, k in enumerate(br):
            delta = mu[i + 1] - mu[i]
            phi = lrv[i + 1] / lrv[i]
            qq = bai_ci_quantiles(phi=phi)
            scale = delta ** 2 / lrv[i]
            cis.append((float(k - qq['q975'] / scale), float(k - qq['q025'] / scale)))
    return {'mu': mu, 'lrv': lrv, 'ci': cis, 'segs': segs}


def bp_cv(T=1000, reps=1000):
    """Bai-Perron critical values (mean shift, trimming 0.15, M = 5): taken from ch2_numbers.json if already computed,
    otherwise simulated (bp_critical)."""
    if 'cv' in _BPCV:
        return _BPCV['cv']
    path = os.path.join(HERE, 'ch2_numbers.json')
    if os.path.exists(path):
        N = json.load(open(path))
        if 'bpcv' in N:
            _BPCV['cv'] = N['bpcv']
            return _BPCV['cv']
    _BPCV['cv'] = bp_critical(T=T, reps=reps)
    return _BPCV['cv']


def ci_dates(s, cis, lab):
    return [[lab(s.index[max(0, int(np.floor(a)))]), lab(s.index[min(len(s) - 1, int(np.ceil(b)))])] for a, b in cis]


def plot_bp(ax, s, fit, col, label):
    ax.plot(s.index, s.values, color=col, lw=1.0, label=label)
    for i, (a, b) in enumerate(fit['segs']):
        ax.plot([s.index[a], s.index[b]], [fit['mu'][i]] * 2, color=st.IDAred, lw=2.4,
                label='segment means' if i == 0 else '_nolegend_')
    for i, (lo, hi) in enumerate(fit['ci']):
        ax.axvspan(s.index[max(0, int(np.floor(lo)))], s.index[min(len(s) - 1, int(np.ceil(hi)))], color=st.Amber, alpha=0.3,
                   lw=0, label='95% interval for the break date' if i == 0 else '_nolegend_')


def threshold_grid(q, trim=TRIM):
    """Candidate thresholds: the distinct values of q between its trim and 1 - trim quantiles."""
    qs = np.sort(np.unique(q))
    lo, hi = np.quantile(q, trim), np.quantile(q, 1 - trim)
    return qs[(qs >= lo) & (qs <= hi)]


def tar_ssr(y, X, q, gammas):
    """SSR of the two-regime model y = X b1 1{q <= g} + X b2 1{q > g} + e for each threshold g."""
    out = np.empty(len(gammas))
    for i, g in enumerate(gammas):
        r = q <= g
        out[i] = ols(y[r], X[r])[2] + ols(y[~r], X[~r])[2]
    return out


def tar_wald_robust(Y, X, q, gammas):
    """Heteroskedasticity-robust Wald statistics for b1 = b2 at each threshold; Y may hold several columns (bootstrap
    samples): returns an array (len(gammas), Y.shape[1])."""
    Y = Y if Y.ndim == 2 else Y[:, None]
    out = np.empty((len(gammas), Y.shape[1]))
    for i, g in enumerate(gammas):
        V, B = [], []
        for r in (q <= g, q > g):
            Xr, Yr = X[r], Y[r]
            A = np.linalg.inv(Xr.T @ Xr)
            b = A @ Xr.T @ Yr
            e = Yr - Xr @ b
            meat = np.einsum('tk,tb,tl->bkl', Xr, e ** 2, Xr)
            V.append(A[None] @ meat @ A[None])
            B.append(b)
        d = (B[1] - B[0]).T
        Vs = V[0] + V[1]
        out[i] = np.einsum('bk,bk->b', d, np.linalg.solve(Vs, d[:, :, None])[:, :, 0])
    return out


def hansen_test(y, X, q, B=300, trim=TRIM, seed=SEED, every=1):
    """sup-Wald test of linearity against a two-regime TAR (heteroskedasticity robust) with the heteroskedastic
    fixed-regressor bootstrap of Hansen (1996): y* = e_hat x N(0, 1), regressors held fixed."""
    gammas = threshold_grid(q, trim)[::every]
    W = tar_wald_robust(y, X, q, gammas)[:, 0]
    _, e0, _ = ols(y, X)
    rng = np.random.default_rng(seed)
    Ys = e0[:, None] * rng.standard_normal((len(y), B))
    Wb = tar_wald_robust(Ys, X, q, gammas).max(axis=0)
    return {'supW': float(W.max()), 'p': float(np.mean(Wb >= W.max())), 'B': B, 'cv5': float(np.quantile(Wb, 0.95))}


def tar_estimate(y, X, q, trim=TRIM):
    """Least-squares threshold, regime estimates, the LR statistic LR(g) = n (S(g) - S(g_hat)) / S(g_hat) with the
    heteroskedasticity adjustment of Hansen (1997, 2000) and the 95% interval {g: LR*(g) <= 7.35} (convexified)."""
    gammas = threshold_grid(q, trim)
    S = tar_ssr(y, X, q, gammas)
    i = int(np.argmin(S))
    g = float(gammas[i])
    n = len(y)
    r = q <= g
    b1, e1, _ = ols(y[r], X[r])
    b2, e2, _ = ols(y[~r], X[~r])
    e = np.empty(n)
    e[r], e[~r] = e1, e2
    s2 = S[i] / n
    LR = n * (S - S[i]) / S[i]
    dx = (X @ (b2 - b1)) ** 2
    Zq = np.column_stack([np.ones(n), q, q ** 2])
    g1 = np.array([1, g, g ** 2]) @ ols(dx, Zq)[0]
    g2 = np.array([1, g, g ** 2]) @ ols(dx * e ** 2, Zq)[0]
    eta2 = float(max(g2 / (g1 * s2), 1e-6)) if g1 > 0 else 1.0
    LRs = LR / eta2
    c = -2 * np.log(1 - np.sqrt(0.95))
    inside = gammas[LRs <= c]
    return {'gamma': g, 'ssr': float(S[i]), 'b1': b1, 'b2': b2, 'n1': int(r.sum()), 'n2': int((~r).sum()), 'eta2': eta2,
            'ci': [float(inside.min()), float(inside.max())], 'gammas': gammas, 'LR': LRs, 'crit': float(c), 'e': e}


def lynx_setar():
    """Tong and Lim (1980): SETAR(2; 7, 2) for log10 lynx, delay 2: AR(7) when y_{t-2} <= r, AR(2) otherwise."""
    y = lynx()
    x = y.values
    p1, p2, d = LYNX_SPEC
    P = max(p1, p2, d)
    yy = x[P:]
    X1 = np.column_stack([np.ones(len(yy))] + [x[P - j:len(x) - j] for j in range(1, p1 + 1)])
    X2 = np.column_stack([np.ones(len(yy))] + [x[P - j:len(x) - j] for j in range(1, p2 + 1)])
    q = x[P - d:len(x) - d]
    gammas = threshold_grid(q, 0.10)
    S = []
    for g in gammas:
        r = q <= g
        S.append(ols(yy[r], X1[r])[2] + ols(yy[~r], X2[~r])[2])
    S = np.array(S)
    g = float(gammas[np.argmin(S)])
    r = q <= g
    b1 = ols(yy[r], X1[r])[0]
    b2 = ols(yy[~r], X2[~r])[0]
    _, X11 = lags(x, 11)
    ar11 = ols(x[11:], X11)
    r0 = q <= 3.116                                       # the threshold of Tong and Lim (1980)
    b1_tl, _, s1 = ols(yy[r0], X1[r0])
    b2_tl, _, s2 = ols(yy[~r0], X2[~r0])
    n = len(yy)
    aic_setar = n * np.log(S.min() / n) + 2 * (p1 + p2 + 2 + 1)
    # skeleton: iterate the deterministic map from the last observations
    path = list(x[-P:])
    for _ in range(60):
        lagv = np.array(path[::-1][:P])
        if lagv[d - 1] <= g:
            path.append(b1[0] + b1[1:] @ lagv[:p1])
        else:
            path.append(b2[0] + b2[1:] @ lagv[:p2])
    sk = np.array(path[P:])
    tail = sk[-30:]
    peaks = [i for i in range(1, len(tail) - 1) if tail[i] > tail[i - 1] and tail[i] >= tail[i + 1]]
    period = float(np.mean(np.diff(peaks))) if len(peaks) > 1 else None
    return {'y': y, 'gamma': g, 'b1': b1, 'b2': b2, 'n1': int(r.sum()), 'n2': int((~r).sum()), 'ssr': float(S.min()),
            'skeleton': sk, 'period': period, 'aic_setar': float(aic_setar), 'b1_tl': b1_tl, 'b2_tl': b2_tl,
            'ssr_tl': float(s1 + s2), 'n': n,
            'resvar_setar': float(S.min() / n), 'resvar_ar11': float(ar11[2] / len(ar11[1]))}


def unemp_tar_data(start, end, p=12, d=12):
    """Hansen (1997): dy_t on a constant and dy_{t-1..t-p}; threshold variable q_{t-1} = y_{t-1} - y_{t-d}."""
    u = unemp_men(True).loc[start:end]
    y = u.values
    dy = np.diff(y)
    t0 = max(p, d)                           # first usable index in dy (needs y_{t-d}: dy index t <-> y index t+1)
    rows = np.arange(t0, len(dy))
    X = np.column_stack([np.ones(len(rows))] + [dy[rows - j] for j in range(1, p + 1)])
    q = y[rows] - y[rows + 1 - d]            # y_{t-1} - y_{t-d}, with y index of dy_t equal to t + 1
    return dy[rows], X, q, u.index[rows + 1]


def logistic(s, g, c, sd):
    return 1 / (1 + np.exp(-np.clip(g * (s - c) / sd, -50, 50)))


def lm3_test(y, Xlin, W, s):
    """LM-type linearity test of Luukkonen, Saikkonen and Terasvirta (1988), F version: regress the residuals of the
    linear model on Xlin and W*s, W*s^2, W*s^3 (W: the regressors allowed to switch, without the constant).
    Also the Terasvirta (1994) sequence H04, H03, H02 (p-values)."""
    _, e, ssr0 = ols(y, Xlin)
    n = len(y)
    Z = [Xlin]
    out = {}
    aux = []
    for k in (1, 2, 3):
        aux.append(W * (s ** k)[:, None])
    Za = np.column_stack(Z + aux)
    _, _, ssr1 = ols(e, Za)
    m = 3 * W.shape[1]
    F = ((ssr0 - ssr1) / m) / (ssr1 / (n - Za.shape[1]))
    out['F'] = float(F)
    out['p'] = float(stats.f.sf(F, m, n - Za.shape[1]))
    # Terasvirta sequence: H04: b3 = 0; H03: b2 = 0 | b3 = 0; H02: b1 = 0 | b2 = b3 = 0
    def ftest(Zr, Zu, mm):
        s0 = ols(e, Zr)[2]
        s1 = ols(e, Zu)[2]
        return float(stats.f.sf(((s0 - s1) / mm) / (s1 / (n - Zu.shape[1])), mm, n - Zu.shape[1]))
    k = W.shape[1]
    out['p4'] = ftest(np.column_stack([Xlin, aux[0], aux[1]]), Za, k)
    out['p3'] = ftest(np.column_stack([Xlin, aux[0]]), np.column_stack([Xlin, aux[0], aux[1]]), k)
    out['p2'] = ftest(Xlin, np.column_stack([Xlin, aux[0]]), k)
    return out


def vdtf_data(end=VDTF_SAMPLE[2], start_est='1970-01-01', p=15, d=1):
    """van Dijk, Terasvirta and Franses (2002, Section 7): dy_t on a constant, 11 monthly dummies, y_{t-1} and
    dy_{t-1..t-15}; transition variable s_t = y_{t-d} - y_{t-d-12}; estimation sample 1970.1-1989.12."""
    u = unemp_men(False).loc[VDTF_SAMPLE[0]:VDTF_SAMPLE[1]]
    y = u.values
    dates = u.index
    dy = np.r_[np.nan, np.diff(y)]
    t = np.arange(len(y))
    rows = t[(dates >= pd.Timestamp(start_est)) & (dates <= pd.Timestamp(end))]
    D = np.column_stack([(dates.month[rows] == mo).astype(float) for mo in range(1, 12)])
    Wm = np.column_stack([y[rows - 1]] + [dy[rows - j] for j in range(1, p + 1)])
    Xc = np.column_stack([np.ones(len(rows)), D])
    s = y[rows - d] - y[rows - d - 12]
    return dy[rows], Xc, Wm, s, dates[rows]


def star_fit(y, Xc, W, s, g0=None, c0=None):
    """LSTAR by nonlinear least squares, linear parameters concentrated out: y = Xc a + W b1 (1 - G) + W b2 G + e,
    G = logistic(gamma (s - c) / sd(s)). Grid over (gamma, c), then a local search."""
    sd = s.std()

    def ssr(par):
        g, c = np.exp(par[0]), par[1]
        G = logistic(s, g, c, sd)
        Z = np.column_stack([Xc, W * (1 - G)[:, None], W * G[:, None]])
        return ols(y, Z)[2]
    best = None
    for lg in np.log([1, 2, 5, 10, 20, 40, 80]):
        for c in np.quantile(s, np.linspace(0.1, 0.9, 17)):
            v = ssr([lg, c])
            if best is None or v < best[0]:
                best = (v, lg, c)
    res = optimize.minimize(ssr, [best[1], best[2]], method='Nelder-Mead', options={'xatol': 1e-4, 'fatol': 1e-8, 'maxiter': 2000})
    g, c = float(np.exp(res.x[0])), float(res.x[1])
    G = logistic(s, g, c, sd)
    Z = np.column_stack([Xc, W * (1 - G)[:, None], W * G[:, None]])
    b, e, S = ols(y, Z)
    return {'gamma': g, 'c': c, 'sd': float(sd), 'ssr': float(S), 'b': b, 'e': e, 'G': G, 'k': Z.shape[1] + 2}


def star_predict(par, Xc, W, s):
    G = logistic(s, par['gamma'], par['c'], par['sd'])
    Z = np.column_stack([Xc, W * (1 - G)[:, None], W * G[:, None]])
    return Z @ par['b']


def estar_resid(par, q):
    th2, mu = par
    x = q[:-1] - mu
    return np.diff(q) + (1 - np.exp(-th2 * x ** 2)) * x


def estar_fit(q):
    """Taylor, Peel and Sarno (2001, eq. 8 with beta1 = -beta1* = 1): q_t - mu = (q_{t-1} - mu) exp(-theta^2 (q_{t-1} - mu)^2)
    + e_t, by nonlinear least squares; standard errors from the Jacobian."""
    best = None
    for th in (0.05, 0.2, 0.5, 1, 2, 5):
        for mu in np.quantile(q, [0.2, 0.35, 0.5, 0.65, 0.8]):
            r = optimize.least_squares(estar_resid, [th, mu], args=(q,), bounds=([1e-6, -5], [100, 5]))
            if best is None or r.cost < best.cost:
                best = r
    e = best.fun
    n = len(e)
    s2 = e @ e / (n - 2)
    J = best.jac
    V = s2 * np.linalg.inv(J.T @ J)
    return {'theta2': float(best.x[0]), 'mu': float(best.x[1]), 'se': np.sqrt(np.diag(V)).tolist(), 's': float(np.sqrt(s2)),
            'n': n, 'ssr': float(e @ e), 'e': e}


def estar_girf(th2, mu, s, shocks=(0.01, 0.05, 0.10, 0.20, 0.30, 0.40), H=60, reps=2000, seed=SEED):
    """Generalised impulse responses from equilibrium (q_{t-1} = mu): mean difference between paths with and without an
    initial shock, same innovations (Koop, Pesaran and Potter 1996; Taylor, Peel and Sarno 2001, Appendix); half-life =
    first horizon at which the response is at most half the shock."""
    rng = np.random.default_rng(seed)
    E = rng.normal(0, s, (reps, H))
    out = {}
    for k in shocks:
        a = np.zeros(reps) + 0.0
        b = np.zeros(reps) + k
        diff = np.empty(H)
        for h in range(H):
            diff[h] = np.mean(b - a)
            a = a * np.exp(-th2 * a ** 2) + E[:, h]
            b = b * np.exp(-th2 * b ** 2) + E[:, h]
        hl = int(np.argmax(diff <= k / 2)) if (diff <= k / 2).any() else None
        out[f'{k:.2f}'] = {'irf': (diff / k).tolist(), 'half_life': hl}
    return out


def estar_girf_history(th2, mu, s, q, shocks=(0.01, 0.05, 0.10, 0.20, 0.30, 0.40), H=80, reps=200, seed=SEED):
    """Generalised impulse responses conditional on the average history (Taylor, Peel and Sarno 2001, Appendix): for every
    observed q_{t-1}, the deviation |q_{t-1} - mu| is the starting point of reps paths with and without a positive shock;
    the responses are averaged over all histories."""
    rng = np.random.default_rng(seed)
    x0 = np.abs(np.asarray(q[:-1]) - mu)
    n = len(x0)
    E = rng.normal(0, s, (n * reps, H))
    start = np.repeat(x0, reps)
    out = {}
    for k in shocks:
        a, b = start.copy(), start + k
        diff = np.empty(H)
        for h in range(H):
            diff[h] = np.mean(b - a)
            a = a * np.exp(-th2 * a ** 2) + E[:, h]
            b = b * np.exp(-th2 * b ** 2) + E[:, h]
        hl = int(np.argmax(diff <= k / 2)) if (diff <= k / 2).any() else None
        out[f'{k:.2f}'] = {'irf': (diff / k).tolist(), 'half_life': hl}
    return out


def df_power_estar(th2, s, T=288, reps=2000, seed=SEED):
    """Rejection rate of the 5% Dickey-Fuller test (constant, no augmentation) when the data follow the estimated ESTAR
    (Taylor, Peel and Sarno 2001, Table 4a)."""
    from statsmodels.tsa.stattools import adfuller
    rng = np.random.default_rng(seed)
    rej = 0
    for _ in range(reps):
        x = np.zeros(T + 100)
        e = rng.normal(0, s, T + 100)
        for t in range(1, T + 100):
            x[t] = x[t - 1] * np.exp(-th2 * x[t - 1] ** 2) + e[t]
        r = adfuller(x[100:], maxlag=0, regression='c', autolag=None)
        rej += r[0] < r[4]['5%']
    return rej / reps


def kss_stat(q, demean=True):
    """Kapetanios, Shin and Snell (2003): t statistic of delta in dq_t = delta q_{t-1}^3 + e_t (demeaned q)."""
    x = q - q.mean() if demean else q
    z, w = np.diff(x), x[:-1] ** 3
    d = (w @ z) / (w @ w)
    e = z - d * w
    se = np.sqrt(e @ e / (len(z) - 1) / (w @ w))
    return float(d / se)


def kss_null(T, reps=2000, seed=SEED):
    """Null distribution of the KSS statistic (demeaned data, random walk), by simulation."""
    rng = np.random.default_rng(seed)
    return np.array([kss_stat(np.cumsum(rng.standard_normal(T))) for _ in range(reps)])


def keenan_test(y, p):
    """Keenan (1985): regress the squared fitted values of the AR(p) on the regressors, then the AR residuals on the
    residual of that regression (one-degree-of-freedom F test)."""
    yy, X = lags(y, p)
    b, e, ssr0 = ols(yy, X)
    f2 = (X @ b) ** 2
    _, v, _ = ols(f2, X)
    eta = (v @ e) / (v @ v)
    ssr1 = ssr0 - eta ** 2 * (v @ v)
    n = len(yy)
    F = (ssr0 - ssr1) / (ssr1 / (n - 2 * p - 2))
    return float(F), float(stats.f.sf(F, 1, n - 2 * p - 2))


def tsay_test(y, p):
    """Tsay (1986): all cross products y_{t-i} y_{t-j} (i <= j <= p), orthogonalised on the AR regressors, explain the AR
    residuals? F test with p(p + 1)/2 restrictions."""
    yy, X = lags(y, p)
    b, e, ssr0 = ols(yy, X)
    L = X[:, 1:]
    C = np.column_stack([L[:, i] * L[:, j] for i in range(p) for j in range(i, p)])
    R = C - X @ np.linalg.lstsq(X, C, rcond=None)[0]
    _, e1, ssr1 = ols(e, R)
    m = C.shape[1]
    n = len(yy)
    F = ((ssr0 - ssr1) / m) / (ssr1 / (n - p - m - 1))
    return float(F), float(stats.f.sf(F, m, n - p - m - 1))


def bds_test(e, m=3, eps=1.0):
    """BDS statistic (Brock, Dechert, Scheinkman and LeBaron 1996) for embedding dimension m, epsilon = eps x sd."""
    from statsmodels.tsa.stattools import bds
    e = np.asarray(e, float)
    st_, p = bds(e, max_dim=m, epsilon=eps * e.std())
    return float(np.atleast_1d(st_)[-1]), float(np.atleast_1d(p)[-1])


def nl_battery(y, p, d=1, B=200):
    """BDS on AR(p) residuals, Keenan, Tsay, LM3 with s = y_{t-d}, Hansen sup-Wald with q = y_{t-d}."""
    y = np.asarray(y, float)
    yy, X = lags(y, p)
    _, e, _ = ols(yy, X)
    s = X[:, d]
    out = {'n': len(yy), 'p': p}
    out['bds'], out['bds_p'] = bds_test(e[-2000:])
    out['keenan'], out['keenan_p'] = keenan_test(y, p)
    out['tsay'], out['tsay_p'] = tsay_test(y, p)
    lm = lm3_test(yy, X, X[:, 1:], s)
    out['lm3'], out['lm3_p'] = lm['F'], lm['p']
    ht = hansen_test(yy, X, s, B=B, every=2)
    out['supW'], out['supW_p'] = ht['supW'], ht['p']
    return out


def fig_nonlinearity_tests(save_it=True, B=200):
    """The test battery on five series: lynx, US unemployment changes, Romanian GDP growth, the real dollar-sterling rate,
    daily EUR/RON returns."""
    series = {
        'log10 lynx (AR(2), d = 2)': (lynx().values, 2, 2),
        'US unemployment, men 20+, change (AR(4))': (np.diff(unemp_men(True).loc[HANSEN_SAMPLE[0]:'2019-12-01'].values), 4, 1),
        'Romanian GDP growth, q/q (AR(1))': (np.diff(100 * np.log(read_eurostat(*GDP_RO).values)), 1, 1),
        'real dollar-sterling rate (AR(1))': (real_usd_gbp().values, 1, 1),
        'EUR/RON daily returns, 2019-2026 (AR(1))': (log_returns('eurron', '2019-01-01').values, 1, 1),
    }
    res = {k: nl_battery(v[0], v[1], v[2], B=B) for k, v in series.items()}
    tests = [('bds_p', 'BDS (m = 3)'), ('keenan_p', 'Keenan'), ('tsay_p', 'Tsay'), ('lm3_p', 'LM3 (Terasvirta)'),
             ('supW_p', 'sup-Wald (Hansen)')]
    M = np.array([[res[k][t] for t, _ in tests] for k in series])
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.imshow(-np.log10(np.maximum(M, 1e-4)), cmap='Blues', vmin=0, vmax=4.5, aspect='auto')
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            v = M[i, j]
            ax.text(j, i, '<0.001' if v < 0.001 else f'{v:.3f}', ha='center', va='center', fontsize=10,
                    color='white' if v < 0.01 else st.DarkText)
    ax.set_xticks(range(len(tests)))
    ax.set_xticklabels([t for _, t in tests])
    ax.set_yticks(range(len(series)))
    ax.set_yticklabels(list(series))
    ax.tick_params(length=0)
    for sp in ax.spines.values():
        sp.set_visible(False)
    plt.tight_layout()
    save('ats_ch2_nonlinearity_tests', save_it)
    return {k: v for k, v in zip(series, res.values())}

## Run

In [ ]:
print(fig_nonlinearity_tests(B=100))

![ats_ch2_nonlinearity_tests](./ats_ch2_nonlinearity_tests.png)

Output: `ats_ch2_nonlinearity_tests.pdf`